# Grondwatermonitoring (GM) in samenhang
Downloading groundwater data for a large area using the different methods in {ref}`gmw-section` can require a lot of requests to the BRO-REST-service: characteristics, GrounwaterMonitoringWells and GroundwaterLevelDossiers/GroundwaterAnalysisReports. To simplify this process PDOK hosts the dataset "Grondwatermonitoring (GM) in samenhang", which contains summary and characteristic information about registration objects within the groundwater monitoring domain, including the relationships between these registration objects.

In {ref}'gm-in-gmw-section' we demonstrated the use of this dataset tod download groundwater heads within a specific area. In this notebook we will show more options to use this dataset, for example to filter the monitoring tubes before downloading actual measurements.

In [ ]:
import matplotlib.pyplot as plt
import brodata

## Download location of tubes
First we will download the location of grundwater monitoring tubes by calling `brodata.gm.get_data_in_extent` with the argument `kind=None`. This only requires a singe request to the PDOK-webservice, and resurns a GeoDataFrame. The index of this GeoDataFrame contains the bro-id of the Groundwater Monitoring Well and the tube number.

In [ ]:
extent = [117700, 118700, 439400, 440400]
tubes = brodata.gm.get_data_in_extent(extent, kind=None)
tubes

## Making a selection of tubes
Before we will download measurements, we can make a selection. For example, we can select all tubes where the top of the well screen is below -10 m NAP.

We can then plot the selection of this GeoDataFrame on a map using `tubes.plot()`.

In [ ]:
max_screen_top_position = -2.5
mask = tubes['screen_top_position'] < max_screen_top_position
print(f"{mask.sum()} of {len(tubes)} tubes selected where screen_top_position < {max_screen_top_position}")
tubes_sel = tubes[mask]

f, ax = plt.subplots()
ax.axis("scaled")
ax.axis(extent)
tubes.plot(ax=ax)
tubes_sel.plot(ax=ax, marker='o', edgecolor='red', facecolor='none');

## Downloading groundwater level data
We can then download groundwater level data using `brodata.gm.get_observations()`. This downloads data for two of the three monitoring tubes. So the third monitoring tube does not contain any groundwater level data.

In [ ]:
obs_df = brodata.gm.get_observations(extent=extent, tubes=tubes_sel, kind="gld", as_csv=True)
obs_df

## Combining tubes and observations
We can then add these observations to the tubes-GeoDataFrame using the method `brodata.gmw.add_observations_to_tubes`. If multiple Groundwater Level Dossiers are available for a single monitoring tube, they are combined.

The index of the tubes-GeoDataFrame and the measurement-dataframe both needs to be a MultiIndex, with the bro-id of the GroundwaterMonitoringWell and the tube-number as the levels. For the tubes-GeoDataFrame this is allready the case, but for the measurement-Dataframe (`obs_df`) we need to change the index.

THe resulting GeoDataFrame contains all the columns from the tubes-GeoDataFrame, with the added columns `observation` and `groundwaterLevelDossier`. The column `observation` contains one DataFrame with observations per monitoring tube. THe column `groundwaterLevelDossier` contains a list of GroundwaterLevelDossier-ids per monitoring tube. If we would have downloaded GroundwaterAnalysisReports (`kind="gar"`), these columns would have been named `laboratoryAnalysis` and `groundwaterAnalysisReport`.

In [ ]:
obs_df = obs_df.reset_index().set_index(["groundwaterMonitoringWell", "tubeNumber"])
tubes_sel = brodata.gmw.add_observations_to_tubes(tubes_sel, obs_df, kind="gld")
tubes_sel

In [ ]:
f, ax = plt.subplots(ncols=2, figsize=(10, 5))
ax[0].axis("scaled")
ax[0].axis(extent)
for i, index in enumerate(tubes_sel.index):
    color = f"C{i}"
    obs = tubes_sel.at[index, "observation"]
    marker = "o"
    if obs.empty:
        marker = "x"
    else:
        obs["value"].plot(ax=ax[1], color=color, label=f"{index[0]} filter {index[1]}")
    tubes_sel.loc[[index]].plot(ax=ax[0], marker=marker, color=color)
    ax[0].annotate(
        f"{index[0]}\n filter {index[1]}",
        xy=tubes_sel.loc[index].geometry.centroid.coords[0],
        ha="center",
        xytext=(0, 5),
        textcoords="offset points",
    )

ax[1].legend();